# Lab 10

STAT 109: Introductory Biostatistics

> **Open in Google Colab (R)**
>
> [Open in
> Colab](https://colab.research.google.com/github/roverhol/stat109-public/blob/main/labs/lab-10.ipynb),
> after it opens, use Runtime → Change runtime type and select R if
> needed.

# Lab 10: Missing data, loading data, and bivariate descriptions

This lab lines up with Lecture 25 (numerical vs categorical), Lecture 26
(two numerical variables), and Lecture 27 (two categorical variables).
By the end you should have the descriptive tools needed for Project 3:
load a TidyTuesday-style dataset, clean missing values sensibly, and
summarize pairs of variables with tables and plots.

------------------------------------------------------------------------

## Learning outcomes

By the end of this lab, you should be able to:

1.  Count and locate missing values (`is.na()`, summaries per column).
2.  Choose a strategy: drop incomplete rows for an analysis, or report
    how many were removed.
3.  Load a CSV from the web with `read_csv()` using a raw GitHub URL
    (TidyTuesday).
4.  Load a CSV you uploaded in Google Colab using the file path (often
    under `/content/`).
5.  For one numerical and one categorical variable: compute group
    summaries and make boxplots (and related plots).
6.  For two numerical variables: make a scatterplot and describe the
    pattern in words.
7.  For two categorical variables: build a two-way table of counts,
    row/column/total proportions, and stacked bar charts.

------------------------------------------------------------------------

## Summary of new functions

| Function / pattern | What it does |
|------------------------------------------|------------------------------|
| `read_csv("url_or_path")` | Reads a CSV from the web or from a file path (e.g. after upload in Colab). |
| `is.na(x)`, `sum(is.na(x))` | Detects missing values; counts how many in a vector. |
| `summarize(across(everything(), ~ …))` | Applies a summary (e.g. count of `NA`) to every column. |
| `drop_na(col1, col2, …)` | Drops rows with missing values in any of the listed columns. |
| `count(A, B)` | Counts rows for each combination of two variables (long format). |
| `pivot_wider()` | Turns long counts into a two-way table with column names from one variable. |
| `group_by()` + `mutate(prop = n / sum(n))` | Converts cell counts to proportions within each group (row or column, depending on what you group by). |
| `group_by(cat)` + `summarize()` | Numeric summaries of one variable within each category. |
| `ggplot()` + `geom_boxplot()` | Compares a numerical variable across categories. |
| `ggplot()` + `geom_point()` | Scatterplot for two numerical variables. |
| `ggplot()` + `geom_col()` | Stacked bar chart when you map `fill` to a second categorical variable. |
| `geom_col(position = "fill")` | 100% stacked bars (proportions within each bar); often with `scale_y_continuous(labels = scales::label_percent())`. |

------------------------------------------------------------------------

## Setup

In [ ]:
if (!requireNamespace("tidyverse", quietly = TRUE)) {
  install.packages("tidyverse", repos = "https://cloud.r-project.org")
}

library(tidyverse)

------------------------------------------------------------------------

## Part 1, Read a TidyTuesday CSV from the web

Many TidyTuesday datasets are stored on GitHub. Open the dataset’s
folder in the [TidyTuesday
repo](https://github.com/rfordatascience/tidytuesday/tree/master/data),
open the CSV, click Raw, and copy the URL from the address bar. Pass
that string to `read_csv()`.

We use the Palmer Penguins file published for TidyTuesday (same data
idea as the `palmerpenguins` package, but loaded by URL so you practice
the workflow you will use on other weeks).

In [ ]:
penguins <- read_csv(
  "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-07-28/penguins.csv",
  show_col_types = FALSE
)

glimpse(penguins)
head(penguins, 10)

------------------------------------------------------------------------

## Part 2, Identify and handle missing values

### 2A) How many missing values per column?

In [ ]:
penguins |>
  summarize(across(everything(), ~ sum(is.na(.)))) |>
  pivot_longer(everything(), names_to = "variable", values_to = "n_missing") |>
  filter(n_missing > 0)

### 2B) Drop rows missing key variables for an analysis

For example, to compare species and island and use bill measurements,
keep rows that have all of these non-missing:

In [ ]:
pg_clean <- penguins |>
  drop_na(species, island, bill_length_mm, bill_depth_mm)

nrow(penguins)
nrow(pg_clean)

You can instead `filter(!is.na(...))` for the same columns. For Project
3, say in your report whether you dropped rows and how many.

------------------------------------------------------------------------

## Part 3, Manual upload in Google Colab (local CSV)

When you are not using a URL, upload a file and read it from the path
Colab assigns.

1.  Click the folder icon to open the file browser.
2.  Upload your `.csv` (drag-and-drop or Upload).
3.  Hover the file → three dots → Copy path (wording may vary slightly).
4.  Paste the path into `read_csv()` inside quotes. Paths often look
    like `"/content/penguins.csv"` or `"/content/drive/MyDrive/..."` if
    using Drive.

In [ ]:
# Replace with the path you copied from Colab after uploading:
# my_data <- read_csv("/content/your_file_name.csv")

On your own laptop (RStudio or Quarto), use a project-relative path or
File → Import; the idea is the same: `read_csv()` needs the correct path
string.

------------------------------------------------------------------------

## Part 4, Bivariate summaries and plots (all pair types)

Use `pg_clean` from Part 2 so plots are not full of `NA` warnings.

In [ ]:
pg_clean <- penguins |>
  drop_na(species, island, bill_length_mm, bill_depth_mm, body_mass_g)

### 4A) Lecture 25, One numerical and one categorical variable

Summaries by group (`species` is categorical; `bill_length_mm` is
numerical):

In [ ]:
pg_clean |>
  group_by(species) |>
  summarize(
    n = n(), mean_bill_mm = mean(bill_length_mm), sd_bill_mm = sd(bill_length_mm), median_bill_mm = median(bill_length_mm)
  )

Boxplot, distribution of bill length within each species:

In [ ]:
ggplot(pg_clean, aes(x = species, y = bill_length_mm, fill = species)) +
  geom_boxplot(alpha = 0.85, show.legend = FALSE) +
  labs(
    title = "Bill length (mm) by species", x = "Species", y = "Bill length (mm)"
  )

Overlaid histograms (optional; from the same lecture idea as comparing
groups):

In [ ]:
ggplot(pg_clean, aes(x = bill_length_mm, fill = species)) +
  geom_histogram(position = "identity", alpha = 0.45, color = "white", binwidth = 2) +
  labs(title = "Bill length by species", x = "Bill length (mm)", y = "Count")

------------------------------------------------------------------------

### 4B) Lecture 26, Two numerical variables

Scatterplot of `bill_length_mm` vs `bill_depth_mm` (both numerical;
color by species optional):

In [ ]:
ggplot(pg_clean, aes(x = bill_length_mm, y = bill_depth_mm, color = species)) +
  geom_point(alpha = 0.85, size = 2) +
  labs(
    title = "Bill depth vs bill length", x = "Bill length (mm)", y = "Bill depth (mm)"
  )

In a sentence: comment on form (roughly linear or curved), direction
(positive or negative association), and strength (strong to weak), as in
Lecture 26.

------------------------------------------------------------------------

### 4C) Lecture 27, Two categorical variables

Two-way table of counts (`species` × `island`):

In [ ]:
pg_clean |>
  count(species, island) |>
  pivot_wider(names_from = island, values_from = n, values_fill = 0)

Proportions, three denominators

- Joint (each cell as a fraction of all penguins in `pg_clean`).
- Row (within species).
- Column (within island).

In [ ]:
# Joint
pg_clean |>
  count(species, island) |>
  mutate(prop = n / sum(n))

# Row: within species
pg_clean |>
  count(species, island) |>
  group_by(species) |>
  mutate(prop_row = n / sum(n)) |>
  ungroup()

# Column: within island
pg_clean |>
  count(species, island) |>
  group_by(island) |>
  mutate(prop_col = n / sum(n)) |>
  ungroup()

Stacked bar charts

In [ ]:
ct <- pg_clean |> count(species, island)

ggplot(ct, aes(x = island, y = n, fill = species)) +
  geom_col(color = "white") +
  labs(
    title = "Counts: species by island", x = "Island", y = "Count", fill = "Species"
  )

ggplot(ct, aes(x = island, y = n, fill = species)) +
  geom_col(position = "fill", color = "white") +
  scale_y_continuous(labels = scales::label_percent()) +
  labs(
    title = "Species mix on each island (column proportions)", x = "Island", y = "Proportion within island", fill = "Species"
  )

------------------------------------------------------------------------

## Part 5, Second dataset (`mpg`) for extra practice *(optional)*

The `mpg` data frame ships with ggplot2. It is handy for short practice
(no download). Examples: `hwy` vs `class` (numerical vs categorical);
`hwy` vs `cty` (two numerical); `class` vs `drv` (two categorical).

In [ ]:
data("mpg", package = "ggplot2")

ggplot(mpg, aes(x = class, y = hwy)) +
  geom_boxplot() +
  theme(axis.text.x = element_text(angle = 30, hjust = 1)) +
  labs(title = "Highway MPG by vehicle class", x = "Class", y = "Highway MPG")

ggplot(mpg, aes(x = cty, y = hwy)) +
  geom_point(alpha = 0.6) +
  labs(title = "Highway vs city MPG", x = "City MPG", y = "Highway MPG")

mpg |> count(class, drv) |>
  ggplot(aes(x = class, y = n, fill = drv)) +
  geom_col(color = "white") +
  theme(axis.text.x = element_text(angle = 30, hjust = 1)) +
  labs(title = "Counts: drive type by class", x = "Class", y = "Count", fill = "Drive type")

------------------------------------------------------------------------

## Checklist toward Project 3

- [ ] Load data from a URL and/or an uploaded file.
- [ ] Report missingness for the variables you use.
- [ ] Classify each of four variables as numerical or categorical.
- [ ] For each pair you care about, pick methods from Labs 9, 10, and
  Lectures 25, 26, and 27 (tables, summaries, and plots).